In [ ]:
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt
from chronos import ChronosPipeline
import torch
import warnings

warnings.filterwarnings('ignore')

print("1. Connecting to database...")
conn = psycopg2.connect(
    dbname="tsdb_transaction", user="vt_user_readonly",
    password="8lthoYeq5xarlwr3", host="timescale-replica.engr.harbingerplatform.com", port="5432"
)

TARGET_FAULT = 'Brake_Fluid_Leak'
CONTEXT_WINDOW = 14 # 5.5 Days

print(f"2. Searching for the most recent '{TARGET_FAULT}'...")
find_fault_sql = f"""
    SELECT deviceid AS truck_id, 
        CAST(DATE_TRUNC('minute', time) AS TIMESTAMP) AS fault_time,
        COUNT(*) as fault_count
    FROM 
        fault_event 
    WHERE 
        faultname = '{TARGET_FAULT}' 
    GROUP BY 
        truck_id
    HAVING 
        COUNT(*) >= 2
    ORDER BY 
        time DESC 
    LIMIT 1;
"""
fault_df = pd.read_sql_query(find_fault_sql, conn)
target_truck = fault_df.iloc[0]['truck_id']
fault_time = fault_df.iloc[0]['fault_time']

print(f"-> Target Truck: {target_truck} | Fault Time: {fault_time}")

# Extract EVERY signal as raw "Long Format" data
print("3. Extracting ALL raw telemetry (Long Format)...")
extract_raw_sql = f"""
    SELECT 
        bucket_1min AS timestamp, 
        '{target_truck}' AS deviceid, 
        signal AS signalnames, 
        min_value, 
        max_value, 
        first_value, 
        last_value, 
        avg_value
    FROM normalized_signal_1hourly
    WHERE deviceid = '{target_truck}'
      AND bucket_1min >= (CAST('{fault_time}' AS TIMESTAMP) - INTERVAL '{CONTEXT_WINDOW} days')
      AND bucket_1min <= CAST('{fault_time}' AS TIMESTAMP)
    ORDER BY timestamp ASC;
"""
raw_df = pd.read_sql_query(extract_raw_sql, conn)
conn.close()
print(f"-> Extracted {len(raw_df)} rows of raw signal data.")

In [1]:
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt
from chronos import Chronos2Pipeline
import torch
import warnings

warnings.filterwarnings('ignore')

print("1. Connecting to database...")
conn = psycopg2.connect(
    dbname="tsdb_transaction", user="vt_user_readonly",
    password="8lthoYeq5xarlwr3", host="timescale-replica.engr.harbingerplatform.com", port="5432"
)
target_truck = 'b5f98353-c811-484b-ab7f-3b51af4a4713'
fault_time = '2026-03-11 15:31:04'
CONTEXT_WINDOW = 8
# Extract EVERY signal as raw "Long Format" data
print("3. Extracting ALL raw telemetry (Long Format)...")
extract_raw_sql = f"""
    SELECT 
        bucket_1min AS timestamp, 
        '{target_truck}' AS deviceid, 
        signal AS signalnames, 
        min_value, 
        max_value, 
        first_value, 
        last_value, 
        avg_value
    FROM normalized_signal_1min
    WHERE deviceid = '{target_truck}'
      AND bucket_1min >= (CAST('{fault_time}' AS TIMESTAMP) - INTERVAL '{CONTEXT_WINDOW} hours')
      AND bucket_1min <= CAST('{fault_time}' AS TIMESTAMP)
    ORDER BY timestamp ASC;
"""
raw_df = pd.read_sql_query(extract_raw_sql, conn)
conn.close()
print(f"-> Extracted {len(raw_df)} rows of raw signal data.")

2026-04-15 17:20:56.982325: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 17:20:57.025543: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-15 17:20:58.395079: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


1. Connecting to database...
3. Extracting ALL raw telemetry (Long Format)...
-> Extracted 1246122 rows of raw signal data.


In [2]:
print("Pivoting data and enforcing strict temporal grids...")

# 1. Pivot the data: timestamps become rows, signals become columns
melted_df = pd.melt(
    raw_df,
    id_vars=['timestamp', 'deviceid', 'signalnames'],
    value_vars=['min_value', 'max_value', 'first_value', 'last_value', 'avg_value'],
    var_name='metric_type',
    value_name='sensor_value'
)

# 2. Create the unique ID for each sequence
melted_df['item_id'] = melted_df['deviceid'] + '_' + melted_df['signalnames'] + '_' + melted_df['metric_type']

# 3. Create a Group ID so the model knows these 5 metrics belong to the same sensor
melted_df['group_id'] = melted_df['deviceid'] + '_' + melted_df['signalnames']

print("Enforcing continuous hourly grid (Allowing NaNs)...")

# Chronos needs continuous sequences. If a truck was off, we need a row with NaN
def enforce_continuous_grid(group):
    group = group.set_index('timestamp')
    group = group[~group.index.duplicated(keep='last')]
    # Generate a perfect 1 hour grid from the first to last timestamp
    full_range = pd.date_range(start=group.index.min(), end=group.index.max(), freq='min')
    group = group.reindex(full_range)

    # Forward fill indentifiers, leave sensor_value as NaN
    group['item_id'] = group['item_id'].ffill().bfill()
    group['group_id'] = group['group_id'].ffill().bfill()
    return group.rename_axis('timestamp').reset_index()

# Apply the continuous grid logic to every item_id independently
final_df = melted_df.groupby('item_id').apply(enforce_continuous_grid).reset_index(drop=True)
final_df = final_df[['item_id', 'group_id', 'timestamp', 'sensor_value']]

print(f"-> Data Melted. Unique Time-Series Sequences Created: {final_df['item_id'].nunique()}")

Pivoting data and enforcing strict temporal grids...
Enforcing continuous hourly grid (Allowing NaNs)...
-> Data Melted. Unique Time-Series Sequences Created: 19485


In [18]:
import torch
from chronos import Chronos2Pipeline

print("1. Loading Chronos-2 Foundation Model...")
# Load the pre-trained foundation model.
pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2", 
    device_map="cuda" if torch.cuda.is_available() else "cpu",
    torch_dtype=torch.bfloat16,
)

target_truck = 'b5f98353-c811-484b-ab7f-3b51af4a4713'
PREDICTION_LENGTH = 60 # Predict the next 48 hours
target_signal = 'VCU_CWTI_Motor_Temp' # The signal we want Chronos to predict

# Isolate a specific signal group to backtest (e.g., Brake Pedal Pos)
# We use the group_id so we grab the min, max, first, last, and avg simultaneously
TARGET_GROUP = f"{target_truck}_{target_signal}"
signal_data = final_df[final_df['group_id'] == TARGET_GROUP]

if len(signal_data) == 0:
    print(f"\n[ERROR] No data found for group: {TARGET_GROUP}")
    print("Here is a list of VALID signals you can copy/paste into 'target_signal':")
    # Print the unique signal names found in the data, removing the deviceid prefix
    valid_signals = [group.replace(f"{target_truck}_", "") for group in final_df['group_id'].unique()]
    print(valid_signals[:20]) # Print first 20 to avoid spamming the console
else:
    context_tensors = []
    item_ids = []

    print(f"2. Preparing Multi-variate Context for {TARGET_GROUP}...")
    for item_id, group in signal_data.groupby('item_id'):
        # Withhold the final 60 minutes for backtesting
        history = group.iloc[:-PREDICTION_LENGTH]['sensor_value'].values
        
        # Chronos handles NaNs natively if passed as floats
        context_tensors.append(torch.tensor(history, dtype=torch.float32))
        item_ids.append(item_id)

    print("3. Executing Zero-Shot Forecasting...")
    # Chronos-2 shares information across all continuous series present within the supplied inference batch [cite: 84]
    forecast = pipeline.predict(
        inputs=context_tensors,
        prediction_length=PREDICTION_LENGTH,
        unrolled_quantiles=[0.1, 0.5, 0.9]
    )

    print("-> Inference Complete!")

1. Loading Chronos-2 Foundation Model...
2. Preparing Multi-variate Context for b5f98353-c811-484b-ab7f-3b51af4a4713_VCU_CWTI_Motor_Temp...
3. Executing Zero-Shot Forecasting...
-> Inference Complete!


In [23]:
import matplotlib.pyplot as plt
import numpy as np

# Extract the specific forecast for the 'avg' metric
avg_index = item_ids.index(f"{TARGET_GROUP}_avg_value")
avg_forecast = forecast[avg_index].numpy() # Shape is now (60, 3)

# Extract the quantiles from the columns!
p10 = avg_forecast[:, 0] # 10th Percentile (Lower Bound)
p50 = avg_forecast[:, 1] # 50th Percentile (Median Baseline)
p90 = avg_forecast[:, 2] # 90th Percentile (Upper Bound)

# Get the ground truth data for those final 60 minutes we withheld
truth_df = signal_data[signal_data['item_id'] == f"{TARGET_GROUP}_avg_value"].iloc[-PREDICTION_LENGTH:]
time_axis = truth_df['timestamp']